# Interpolation of LORA outputs from sigma to z coordinates
A few notes:
- land masks are zeros.
- The bottom values (z = -1) are all zeros, which need to be NaNs for interpolation to work properly.

In [6]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
from numba import njit, prange
import pandas as pd

In [17]:
list_in_file

['npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 'npac.2017',
 '01.nc',
 '02.nc',
 '03.nc',
 '04.nc',
 '05.nc',
 '06.nc',
 '07.nc',
 '08.nc',
 '09.nc',
 '10.nc',
 '11.nc',
 '12.nc']

### Interpolation from sigma-coordinate to z-coordinate

In [18]:
# Define Target Z-Levels (meters)
z_targets = np.array([
    0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 
    60, 70, 80, 90, 100, 125, 150, 175, 200, 
    250, 300, 350, 400, 450, 500, 600, 700, 800, 900, 1000
], dtype=np.float32)

# Define your list of variables
list_var = [
    {
        "outname": "swr",
        "varname": "swr",
        "units": "W/m^2",
        "cfname": "shortwave_radiation"
    },
    {
        "outname": "t",
        "varname": "t",
        "units": "degC",
        "cfname": "sea_water_temperature"
    },
    {
        "outname": "s",
        "varname": "s",
        "units": "psu",
        "cfname": "sea_water_salinity"
    },
    {
        "outname": "u",
        "varname": "u",
        "units": "m/s",
        "cfname": "eastward_sea_water_velocity"
    },
    {
        "outname": "v",
        "varname": "v",
        "units": "m/s",
        "cfname": "northward_sea_water_velocity"
    },
    {
        "outname": "k",
        "varname": "kh",
        "units": "m2/s",
        "cfname": "ocean_vertical_tracer_diffusivity"
    }
]

# Define the experiment directory
dir_exp = '/snow/hakaseh/py-off-bgc/input/LORA'
exp_name = 'LORA'

# Loop through years
for year in range(2017,2024):
    # input file name
    prefix_in = ["npac."+str(year)] * 12
    list_in_file = np.array(prefix_in) + np.array([
                    "01.nc",
                    "02.nc",
                    "03.nc",
                    "04.nc",
                    "05.nc",
                    "06.nc",
                    "07.nc",
                    "08.nc",
                    "09.nc",
                    "10.nc",
                    "11.nc",
                    "12.nc"
                   ])

    # Loop through months
    for in_file in list_in_file:
        # 1. Load dataset
        ds = xr.open_dataset(f"{dir_exp}/{in_file}")
        
        # Bathymetry (H)
        H = ds['h']
        # zz in LORA are fractions so multiply by H
        z_static = abs(ds['zz'])*H  # zz values are negatives, so use abs() to convert to positives
        
        el = ds['el']
        
        # Calculate ACTUAL 3D Depth for each day
        #    Formula: Z_static * (1 + EL / H)
        z_daily = z_static * (1.0 + el / H)
        z_daily = z_daily.transpose("time","z","y","x")
        
        # Use salinity data to create land mask (zeros are land, including the bottom)
        da_mask3d = ds["s"].isel(time=0).where(ds["s"].isel(time=0) == 0, other = 1)
        da_mask3d = da_mask3d.where(da_mask3d == 1, other = np.nan)
        
        @njit(parallel=True)
        def interpolate_fast(z3d_in, in_da, z_targets, out_da):
            nz, nj, ni = in_da.shape
            
            # Numba likes explicit loops! 
            # Use 'prange' on the outer loop to use all CPU cores automatically
            for j in prange(nj):
                for i in range(ni):
                    # Extract columns for depth and data (Standard numpy slicing works in Numba)
                    col_z = z3d_in[:, j, i]
                    col_da = in_da[:, j, i]
                    
                    # Numba supports np.interp
                    out_da[:, j, i] = np.interp(z_targets, col_z, col_da)
        
        for v in list_var:
            print(f"Interpolating {v['varname']}...")
        
            # Read the input data
            da_in = ds[v["varname"]]
            nt, ny, nx = da_in["time"].size, da_in["y"].size, da_in["x"].size
        
            if da_in.ndim == 3: 
                print("2D spatial data (no depth). no interpolation. just cleaning...")
                da_out = da_in.values*da_mask3d.isel(z=0).values
        
                out_array = xr.DataArray(
                    da_out,
                    coords={
                        "time": ds["time"],
                        "lat": ds["north_e"][:,0].values,   # Copy lat from the grid file
                        "lon": ds["east_e"][0,:].values,    # Copy lon from the grid file
                    },
                    dims=("time", "lat", "lon"),
                    name=v["outname"],
                    attrs={
                        "units": v["units"],
                        "standard_name": v["cfname"]
                    }
                )            
            elif da_in.ndim == 4:
                print("3D spatial data. vertical interpolation...")
                # Initialize the output array
                da_out = np.full((nt, len(z_targets), ny, nx), np.nan, dtype=np.float32)
        
                for t in range(nt):
                    # Call Numba
                    interpolate_fast(z_daily[t,:,:,:].values, 
                                     da_in[t,:,:,:].values*da_mask3d.values, 
                                     z_targets, da_out[t,:,:,:])
                    
                # Convert back to Xarray
                out_array = xr.DataArray(
                    da_out,
                    coords={
                        "time": ds["time"],
                        "depth": z_targets,       # New Z coordinates
                        "lat": ds["north_e"][:,0].values,   # Copy lat from the grid file
                        "lon": ds["east_e"][0,:].values,    # Copy lon from the grid file
                    },
                    dims=("time", "depth", "lat", "lon"),
                    name=v["outname"],
                    attrs={
                        "units": v["units"],
                        "standard_name": v["cfname"]
                    }
                )
        
            # Save to Disk
            out_filename = f"../input/{exp_name}/input_{exp_name}_{v['outname']}_{in_file}"
            
            # converting to dataset allows preserving global attributes easily
            ds_out = out_array.to_dataset()
            ds_out.to_netcdf(out_filename)
            
        print (f"Interpolation finished: {in_file}")

Interpolating swr...
2D spatial data (no depth). no interpolation. just cleaning...
Interpolating t...
3D spatial data. vertical interpolation...
Interpolating s...
3D spatial data. vertical interpolation...
Interpolating u...
3D spatial data. vertical interpolation...
Interpolating v...
3D spatial data. vertical interpolation...
Interpolating kh...
3D spatial data. vertical interpolation...
Interpolation finished: npac.201701.nc
Interpolating swr...
2D spatial data (no depth). no interpolation. just cleaning...
Interpolating t...
3D spatial data. vertical interpolation...
Interpolating s...
3D spatial data. vertical interpolation...
Interpolating u...
3D spatial data. vertical interpolation...
Interpolating v...
3D spatial data. vertical interpolation...
Interpolating kh...
3D spatial data. vertical interpolation...
Interpolation finished: npac.201702.nc
Interpolating swr...
2D spatial data (no depth). no interpolation. just cleaning...
Interpolating t...
3D spatial data. vertical int